In [ ]:
import warnings
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
warnings.filterwarnings('ignore')  # Игнорировать все предупреждения (не рекомендуется в продакшн-коде)
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from numpy import mean
from numpy import var
from math import sqrt
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from causalinference import CausalModel
from scipy.stats import chisquare
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import tt_ind_solve_power
from sklearn.metrics import r2_score

**TODO**  
изучить функционал библиотек  
https://github.com/pyro-ppl/pyro  
https://github.com/py-why/dowhy  
https://github.com/uber/causalml  

Ключевая книга  
https://matheusfacure.github.io/python-causality-handbook/01-Introduction-To-Causality.html  

Ссылки на датасеты  
https://github.com/matheusfacure/python-causality-handbook/blob/master/causal-inference-for-the-brave-and-true/data/collections_email.csv  

Чекнуть книгу:  
Inference and Intervention Causal Models for Business Analysis  
    
https://www.kaggle.com/code/harrywang/simple-matching-in-python  
https://habr.com/ru/companies/ods/articles/544208/   
https://harrywang.me/psm-did  
Цикл статей о казуальности: https://towardsdatascience.com/why-do-we-need-causality-in-data-science-aec710da021e  
Методы используемые в Uber: https://www.uber.com/en-RS/blog/causal-inference-at-uber/  
Книга по Causal Inference - https://matheusfacure.github.io/python-causality-handbook/landing-page.html

 
методы изучить:      
Bayesian structural time series  
Bayesian Statistics  

#### CAUSAL INFERENCE - ТЕОРИЯ

<u>ОБЩАЯ ЗАДАЧА</u>  
Оцениваем зависимость Y(T), где Y - целевая метрика, T=treatment - метрика влияния на выборку.  
Выборка - возможные клиенты или др. сущности у которой есть набор признаков X.  
**Пример**: оценка влияния типа лекарства T=0;1 на вероятность излечения пациента Y.  
Распределение T - по сути то, как мы собираем сравниваемые (control;test) группы (их может быть много).  
  
ATE = avg. treatment eff = avg(Y_e - Y_c) в группах (control;test)  
ATE = avg(Y_e|exp) - avg(Y_c|con) = avg(Y_e|exp) - avg(Y_c|exp) + avg(Y_c|exp) - avg(Y_c|con) =  
avg(Y_e - Y_c|exp) - Y_c_exp_con_diff = ATT - BIAS  
ATT = avg eff on treatment = насколько изменим Y если окажем влияние на тестовую группу (до/после).  
BIAS = различие сравниваемых контрольной и тестовой выборок без реального тестового изменения Y=Y_c
  
BIAS>0 когда сравниваемые группы не эквивалетны по сути (выборки не сбалансированы).  
BIAS>0 возникает когда в X присутствуют переменные C - <u>конфаундеры</u>.  
C->T; C->Y => у семплов выборки есть разная вероятность попасть в control;test и разные Y зависящие от C.   
**Пример**: Y=вероятность излечения; T=тип лекарства; С=кол-во денег пациента. Богатые пациенты более вероятно  
получат лучшее лекарство T, а также могут позволить хорошее питание итд. также влияющее на Y. В результате,  
посчитанный ATT будет завышен - не будет предсказывать корректно как реально изменится Y в более бедной группе.  
  
Рандомизированный AB тест - лучшее решение, тогда control; test без влияния симметричны по Y -> BIAS = 0  
Тогда ATE = ATT + 0 = ATT. В этом случае отсутствуют C, т.к рандом разрушает влияние C->T.  
Но это часто невозможно - дорого, исторические данные, неэтично и.т.д  
Тогда работаем не с тестовыми данными, а с наблюдаемыми без искуственного теста (observation data)  

Итак, в тестовых данных - работа больше над чувствительностью критериев детектирования ATE  
В наблюдаемых данных - проблема балансировки (выборки симметричны по переменным, которые влияют на Y)  
Иными словами, ослабление конфаундеров C (если С->Y, то стремимся убрать влияние C->T).  
  
---
<u>ВЕРОЯТНОСТНЫЕ ГРАФЫ</u>  
A->B = событие А влияет на событие В, т.е каждое dA порождает соответствующее dB.  
A->B не значит что B->A (чем холоднее - чаще кашляешь, но если часто кашлять - не станет холоднее)  
  
Основные связи на вероятностных графах:
1. A->B->C = **blocker**. В среднем, A влияет на C (например, больше А -> больше B -> больше С)  
Но при B=const A и C независимы (B - является блокером связи)  
Пример: знаешь статистику -> решаешь задачи интервью -> получаешь хороший оффер.  
2. A->B; A->C = **fork**. В среднем B связан с С (например, больше B -> больше A -> больше С)  
Но при A=const, B и C - независимы.  
Пример: A - знания статистики; B,C - знания эконометрики и ML. В среднем, чем лучше знаешь ML -  
тем больше шансов что знаешь эконометрику через статистику. Но при фиксированном уровне знаний статистики   
рост знаний в ортогональных дисциплинах независим
3. B->C; A->C = **collider**. В общем, В и А - независимы. Но при C=const A и B зависимые  
Пример: C - оффер; B - хард скиллы; С - софт скиллы. Если хороший оффер, но низкие харды -> софты итд.


ДОПОЛНИТЕЛЬНО  
**CACE** = Complier Average Causal Effect  
Пример методики, когда мы проверяем эффект воздействия, которое может быть применено не ко всем в группе.  
Пример: в группе А люди лечатся лекарством 1, при этом достаточно часто его принимают.  
В группе В мы выписываем им новое лекарство - оно сильно более эффективное, но дорогое -> сильно  
реже люди его покупают после рецепта и используют для лечения.  
По итогу, может оказаться, что невыгодно выписывать лекарство В, но если просто сравнивать эффект на  
тех кто его принял - результат будет обратным (конфаундер здесь - цена).  
CACE = (result_B - result_A) / (p_B - p_A), где p - вероятность принять лекарство (изменение).

#### ЛИНЕЙНАЯ РЕГРЕССИЯ

Линейная регрессия позволяет оценивать зависимости для моделей Y = a1 * x1 + a2 * x2 + ... an  
По набору данных Y;X мы оцениваем коэффициенты A (вкл. доверительные интервалы и.т.д)  
a1 - коэф. который отражает как будет X1->Y при фиксированных X2, ... 
  
Построим регрессию Y = a1 * T + a2 * C1 + a3 * C2 + ...; T=treatment, C=confounders  
Тогда мы получим T->Y при С=const - то есть в условиях когда нет фактора C->T.  
Таким образом, если удастся собрать полный набор C - конфаундеров, можно получить честную Y(T)  

<u>Что стоит и не стоит включать в модель X регрессии</u>  
Стоит включать в модель:
1. Все присутствующие конфаундеры C: C->T; C->Y.  
Их учет сделает менее смещенным полученный предикт Y(T).  
2. Предикторы X: X->Y, но при этом X не влияет на T. Они еще называются ковариатами.   
Эти предикторы снижают дисперсию предсказаний Y -> улучшают качество предикта Y(T)  
  
Нельзя включать в модель:
1. Предикторы, которые не влияют на Y
Модель пытается найти фиктивную зависимость, что увеличивает шум оценки  
2. Предикторы-коллайдеры CL: T->CL; Y->CL  
См. графы в разделе "Теория": при CL=const между T;Y индуцируется фиктивная связь -> bias
3. Предикторы-блокеры BL: T->BL->Y
См графы: при BL=const между T;Y искусственно нарушается связь -> bias  
4. Признаки "из будущего" P: Y->P (переобучение)
  
Пункты 2, 3 относятся к проблеме common effect - когда включается несколько переменных оказывающих  
связь на целевую -> возникает неопределенность в атрибуцировании этой связи.  
Проблема корреляции сюда же, так как при corr(T,X)~1 следует в том числе T->X; X->Y. 
  
Итого: нельзя включать в модель X все признаки подряд, необходимо отбирать только  
уточняющие модель признаки, а также конфаундеры. Проблема отбора качественных признаков - 
исключительно в доменных знаниях о проблеме.  
До конца нельзя быть уверенными что X собран оптимально -> всегда есть шанс получить BIAS оценки Y(T).  
Единственный способ проверить качество оценки честно - рандомизированный AB.  

--- 
**Пример коллайдера CL**: Y = доход; T = уровень образования; CL = активность инвестирования.  
Y->CL (больше денег - актуальнее проблема инвестирования); T->CL (образование -> изучение инвестирования).  
Если в модели сделать CL=Const, то мы индуцируем зависимость T->Y даже если ее нет в среднем.  
Люди с высокой инвест-активностью: если не имеют денег то с большкей вероятностью имеют высокое образование  
(иначе они бы не инвестировали) и.т.д

In [ ]:
# влияние факта отправки письма T=email=0;1 на погашение задолженности payments
df = pd.read_csv("./data/collections_email.csv"); print(df.shape[0])
df.head(1)

In [ ]:
# сплит email=0;1 рандомный. Оценка T-тестом говорит, что недостаточно 
# мощности для того чтобы увидеть значимый прокрас в payments
from help_tools import ttest_calc
ttest_calc(df[df.email == 1].payments, df[df.email == 0].payments)[0]

In [ ]:
# аналогично видим что p_value > 0.05 при оценке зависимости payments = f(email)
from help_tools import ols_calc
ols_calc(df, smf_formula='payments ~ email')[0]

In [ ]:
# однако проблема в том числе в том, что email - оч мало влияет на погашение с сравнение с др. факторами
# в частности credit_limit - сколько было взято в долг; risk_score
# мы не можем из за этого "рассмотреть" влияние email; но если зафиксировать эти факторы X->Y, 
# то влияние email получится оценить (получается метод улучшения чувствительности)
ols_calc(df, smf_formula='payments ~ email + credit_limit + risk_score')[0]

In [ ]:
# посмотрим что будет, если включить в систему блокер BL = переменную opened - 
# открытие письма. Оно возможно только когда email=1 т.е email -> opened -> payments.  
# при opened=const (что делает регрессия) => связь email -> payments ослабляется, что приводит к bias
ols_calc(df, smf_formula='payments ~ email + credit_limit + risk_score + opened')[0]

In [ ]:
df = pd.read_csv('./data/medicine_impact_recovery.csv').rename(columns={'medication':'treatment',
                                                                        'recovery':'y'})#.drop(columns='severity')
ols_calc(df, smf_formula='y ~ sex + age + treatment + severity')[0]

<u>ПРИЛОЖЕНИЕ</u>  
Если в регрессии имеется дисбаланс в переменной X -> можно ослаблять влияние значения в  
зависимости от частоты его встречаемости. Для этого есть метод sample_weighted = freq.  
Хорошо подходит для корректировки гетероскедастичности - когда дисперсия переменной Y  
сильно зависит от величины X.  
  
Регуляризация в регрессии - когда полученные веса a1, a2, ... штрафуются за очень высокие значения.  
Так - один признак не будет иметь тенденции перетянуть на себя все объяснение -> снижается риск  
переобучения модели. Типы регуляризации:
1. Lasso = L1 = |a1| + |a2| + ...  
Приводит к занулению некотрых весов - полезная для отбора признаков x1, x2 ... - разреженные модели.  
2. Ridge-регуляризация = L2 = a1^2 + a2^2 + ...   
учитывает все веса с более плавным штрафом. Полезно когда хотим сохранить все признаки в оптимизации,  
вычислительно более дорогая штука.

#### БАЗОВЫЙ МАТЧИНГ

Необходимо оценить зависимость Y(T) в условиях наличия конфаундеров C: C->T;C->Y.  
Помимо прочего учитываем ковариаты X=C+Cv: Cv->Y для снижения дисперсии оценки и роста точности.  
Регрессия решает это оценкой Y(T|X=const). Минус, что полагается линейная зависимость Y = Y(T, X)  
  
Идея матчинга - подбирать схожие по распределению X семплы выборки в разных T, чтобы  
сравниваемые T=0;1 выборки были сбалансированны по признакам. 
  
Один из способов - использовать метод kNN для поиска для каждого семпла из T=1 наиболее близкого из T=0 в  
пространстве признаков X. Здесь нет ограничения на линейность и как следствие близкое (через экстраполяццию)  
сходство признаков семплов в группах T=0;1  

Проблема: проклятие размерности. Чем больше X.columns, тем сложнее найти близкого соседа. Возникает BIAS  
обусловленный тем что X0 не идеально совпадает с X1 (сматченные семплы из групп T=0;1)  
То есть Y0 - Y1 = (разница из за влияния T=1) + (разница из за расхождения dX).  
  
Можно скорректировать влияние dX через регрессию: вместо Y0 строим Yo_forecast(X1) чтобы "пересадить"  
сравниваемую точку X0 в X1 (по значениям но при этом с T=0).  

In [ ]:
# df = pd.read_csv('./data/groupon.csv')[['prom_length', 'price', 'discount_pct', 'coupon_duration', 
#          'featured','limited_supply', 'treatment', 'revenue']].rename(columns={'revenue':'y'})
# df = pd.read_csv('./data/groupon.csv')[['prom_length', 'price', 'discount_pct', 'coupon_duration', 
#          'featured', 'treatment', 'revenue']].rename(columns={'revenue':'y'})
# df = pd.read_csv('./data/smoker.csv').rename(columns={'outcome':'y'})
df = pd.read_csv('./data/medicine_impact_recovery.csv').rename(columns={'medication':'treatment', 'recovery':'y'})
df.head(1)

In [ ]:
# смещенный ATE (не корректен, так как выборки не сбалансированы из за конфаундеров) 
df[df.treatment == 1]['y'].mean() - df[df.treatment == 0]['y'].mean()

In [ ]:
# берет точку x; находит n_neigb ее ближайших соседей в пространстве X; 
# вычисляет и возвращает усредненное Y для этих соседей
from sklearn.neighbors import KNeighborsRegressor
col = 'y'
treated = df[df.treatment == 1]
untreated = df[df.treatment == 0]

X_treated = treated.drop(columns=col)
X_untreated = untreated.drop(columns=col)
y_treated = treated[col].values
y_untreated = untreated[col].values

# просто запомнили координаты всех точек в двух массивах
mt0 = KNeighborsRegressor(n_neighbors=1).fit(X_untreated, y_untreated)
mt1 = KNeighborsRegressor(n_neighbors=1).fit(X_treated, y_treated)

# для каждой точки ищем Y соседа (т.к он 1 то значение точное)
treated['match'] = mt0.predict(X_treated)
untreated['match'] = mt1.predict(X_untreated)

# Корректировка неточного матчинга по X через лин регрессию
treated_match_index = mt0.kneighbors(X_treated, n_neighbors=1)[1].ravel()
untreated_match_index = mt1.kneighbors(X_untreated, n_neighbors=1)[1].ravel()
ols0 = LinearRegression().fit(X_untreated, y_untreated)
ols1 = LinearRegression().fit(X_treated, y_treated)
untreated['bias_correct']=ols1.predict(X_untreated) - ols1.predict(treated.iloc[untreated_match_index].drop(columns=[col, 'match']))
treated['bias_correct']=ols0.predict(X_treated) - ols0.predict(untreated.iloc[treated_match_index].drop(columns=[col, 'match', 'bias_correct']))

# ATE оценка эффекта - берем все пары x-x_match и сравниваем. k = 2*T-1 = 1 if treatment else -1. Чтобы не занулять пары.
df_s = pd.concat([treated, untreated])

# np.mean((2*df_s["treatment"] - 1)*(df_s[col] - df_s["match"])) # оценка без учета bias
np.mean((2*df_s["treatment"] - 1)*(df_s[col] - df_s["match"] - df_s['bias_correct'])) # коррекция dx через регрессию

In [ ]:
# Аналогичный kNN матчинг по X через библиотеку
from causalinference import CausalModel
cm = CausalModel(
    Y=df[col].values, 
    D=df["treatment"].values,  #T
    X=df.drop(columns=[col, 'treatment']).values
)

cm.est_via_matching(matches=1, bias_adj=True)
print(cm.estimates)

In [ ]:
# РЕГРЕССИЯ ДАЕТ СМЕЩЕННЫЕ ОЦЕНКИ, ТАК КАК НЕ ВЫПОЛНЯЮТСЯ ТРЕБОВАНИЯ ЛИНЕЙНОСТИ
# model = smf.ols('y ~ treatment + smoker', data=df).fit()
model = smf.ols('y ~ treatment + sex + age + severity', data=df).fit()
# model = smf.ols('y ~ treatment + prom_length + price + discount_pct + coupon_duration + featured', data=df).fit()
model.summary().tables[1]

Приложение.  
<u>Парадокс Симпсона</u>  
Частный случай несбалансированных выборок.    
Смотрим эффект от лекарства в разбивке по молодым и пожилым. Допустим, в обеих группах лекарство помогает.  
Но если объединить пожилых и молодых и их процентный состав разный в A/B -> может произойти инверсия.  
Причина - вылечиваемость зависит не только от лекарства, но и от возраста. Смещение выборок по возрасту ->  
искажение реального эффекта с контролем на возраст.  
https://towardsdatascience.com/solving-simpsons-paradox-e85433c68d03  
  
Возраст влияет не только на излечиваемость, но и на формирование T=тест/контроль выборки (к примеру,  
более пожилым людям априори дают более проверенное лекарство B, что вносит асимметрию).  
Разрешение парадокса как раз возможно через методы матчинга или контроль X регрессией итд.   

#### PROPENSITY SCORE MATCHING (Учет нелинейности)

Основной источник - https://matheusfacure.github.io/python-causality-handbook/11-Propensity-Score.html  
Дополнительные статьи:  
https://harrywang.me/psm-did  
https://towardsdatascience.com/a-hands-on-introduction-to-propensity-score-use-for-beginners-856302b632ac  
https://www.kaggle.com/code/harrywang/propensity-score-matching-in-python/notebook

---

В случае с регрессией мы контролируем Y=Y(T|X=const)  
Можно контролировать avg(T|X) - то есть чтобы для каждого набора семплов из dX был одинаковый шанс попасть в T=0;1  
В базовом матчинге говорим что для каждой группы семплов с близкими X - половина присутствуем в T=0 и в T=1  
Проклятие размерности - при большом X.columns сложно работать с dX где оч мало семплов.  
Регрессия по сути X->Y_forecast сводит многомерный массив в одну переменную (но с ограничением линейности)  
  
Propensity score method: X -> propensity score = вероятность попасть в группу T=1 = Logistic_regression_prob_T(X)  
Оказывается, Y = Y(T|X=const) = Y(T|ps = const), где ps = ps(X).  
Т.е редуцируем многоразмерный X в одноразмерную вероятность без ограничения общности, как в линейных моделях.  

**Ограничение**: распределения ps(группы T=1) и ps(группы T=0) должны пересекаться, т.е overlap(ps1, ps2)>0.  
Это значит, что для большинства семплов X имеется ненулевая вероятность попасть в любой из классов T=0; 1.  
А значит, найдется похожий зеркальный семпл из другой группы для матчинга.  
Если у семпла ps_i=0, то алгоритм выбрасывает такие точки из рассмотрения.
  
<u>Пример</u> - сравниваем эффективность вакансий Стандарт и Стандарт+. Если Ст+ публикуют только частные компании,  
а Ст - только гос. компании, то ни у одной гос-вакансии нет условной вероятности стать Ст+ (ps=0).  
Значит нельзя подобрать "похожие" вакансии с точки зрения шанса стать Ст/Ст+ для честного сравнения эффективности.  
Нельзя экстраполировать свойства из группы Ст+ на совершенно иные свойства группы Ст.  
  
Приложение:  
Оценка погрешности вычисляемого ATE должна базироваться на точности вычисления ps.  
Здесь есть множество проблем, так как бутстрап работает для матчинга нестабильно.  
Возможно использование регрессии для поиска доверительных интервалов эффекта, но есть ограничения линейности y(ps).  

Приложение 2:  
не стоит включать в систему X при расчете ps ковариаты, которые прямо коррелируют с T  
Это приведет к полной разделимости семплов, то есть к тому что overlap -> 0 и нельзя применять матчинг.

In [ ]:
df = pd.read_csv('./data/groupon.csv')[['prom_length', 'price', 'discount_pct', 'coupon_duration', 'featured', 'treatment', 'revenue']].rename(columns={'revenue':'y'})
# df = pd.read_csv('./data/medicine_impact_recovery.csv').rename(columns={'medication':'treatment', 'recovery':'y'})
# df = pd.read_csv('./data/medicine_impact_recovery.csv').rename(columns={'medication':'treatment', 'recovery':'y'}).drop(columns='severity')

# df = pd.read_csv('./data/smoker.csv').rename(columns={'outcome':'y'})
df.head(2)

In [ ]:
# propensity score calc 
ps_model = LogisticRegression(C=1e6).fit(df.drop(columns=['y', 'treatment']), df.treatment)
df['ps'] = ps_model.predict_proba(df.drop(columns=['y', 'treatment']))[:, 1] # вероятность принадлежности к классу T=1

# overlap check
plt.grid()
plt.title('OVERLAP CHECK')
sns.histplot(data=df, x='ps', hue='treatment')

# теперь можно перейти y(T, X) = y(T, ps)
# матчинг на основе kNN (ищем ближайшего по ps оппонента - сравниваем)
from sklearn.neighbors import KNeighborsRegressor
treated = df[df.treatment == 1]
untreated = df[df.treatment == 0]

mt_tr = KNeighborsRegressor(n_neighbors=1).fit(treated[['ps']], treated.y)
mt_untr = KNeighborsRegressor(n_neighbors=1).fit(untreated[['ps']], untreated.y)
untreated['y_match'] = mt_tr.predict(untreated[['ps']])
treated['y_match'] = mt_untr.predict(treated[['ps']])
dfs = pd.concat([treated, untreated])

# ATE = какой был бы эффект если б каждый элемент всей выборки сконвертился T=0 -> T=1
print(f"""ATE_PSM = {np.mean((2*dfs["treatment"] - 1)*(dfs['y'] - dfs["y_match"]))}""")

In [ ]:
# Аналогичный kNN матчинг по ps через библиотеку
from causalinference import CausalModel
cm = CausalModel(
    Y=df.y.values,
    D=df["treatment"].values, # T
    X=df.ps.values
)
cm.est_via_matching(matches=1, bias_adj=True)
print(cm.estimates)

#### IPTW

IPTW = Inverse probability of treatment weighting  
У нас есть две группы control и test, которые не сбалансированы, т.е распределение X в них разное.  
Это значит что есть такие слои значений dX семлпы с которыми чаще представлены в control и наоборот.  
Можно это рассмотреть так: клиенты со свойствами X~dX с меньшей вероятностью получают T=1.  
Пример: половине пациентов даем лекарство, но из них есть молодые которые меньше его принимают итд.  
  
Оценим ps(X) - шанс получить T=1 когда у семпла вектор признаков X.  
Два подхода к балансировке.  
1. PSM. Для каждого семпла X и T=1 подбираем зеркальный семпл X2: ps(X2)~ps(X) у которого T=0,   
если ps->0 или 1 значит у этого семпла нет пары, выкидываем семплы с экстремальными ps  
2. IPTW. Вес каждого семпла с ps(X) при расчете вклада в метрику Y рассчитываем как w=1/ps|T=0 или 1/(1-ps)|T=0  
То есть чем меньше был шанс оказаться в той группе, где оказался семпл - тем больше его вес.  
Это позволяет выровнять случаи когда для какого то интервала dX почти все семлпы в тесте или контроле.  
При этом мы не выбрасываем данные.  
  
ATE = avg(Y|X,T=1) - avg(Y|X,T=0) = avg(Y * T / ps) - avg(Y * (1 - T) / (1 - ps)) = avg(Y * (T - ps) / ps (1 - ps))  
Напомним, что ATE = эффект от влияния T=1 если бы оно коснулось всех элементов выборки.  
  
Приложение:  
Как и для PSM критично условие overplap>0. Если у семпла ps=0 и он не представлен в выборке T=0, то нельзя его отмасштабировать  
  
Приложение 2:  
PSM работает неплохо на малых выборках, тогда как для IPTW необходимы большие выборки.  
В противном случае, т.к мы учитываем вклад семплов с экстремальными ps ~ 0, будут искажения в результатах.  
  
Реализацию iptw в python см в разделе "Сравнение моделей"

In [ ]:
# df = pd.read_csv('./data/smoker.csv').rename(columns={'outcome':'y'})
df = pd.read_csv('./data/groupon.csv')[['prom_length', 'price', 'discount_pct', 'coupon_duration', 'featured', 'treatment', 'revenue']].rename(columns={'revenue':'y'})
# df = pd.read_csv('./data/medicine_impact_recovery.csv').rename(columns={'medication':'treatment', 'recovery':'y'})
df.head(1)

In [ ]:
# ps calc
ps_model = LogisticRegression(C=1e6).fit(df.drop(columns=['y', 'treatment']), df.treatment)
df['ps'] = ps_model.predict_proba(df.drop(columns=['y', 'treatment']))[:, 1] 
plt.grid()
plt.title('OVERLAP CHECK')
sns.histplot(data=df, x='ps', hue='treatment')
# IPTW calc
weight = ((df.treatment - df.ps) / (df.ps * (1 - df.ps)))
# ATE calc
print(f'ATE_IPTW = {np.mean(weight * df.y)}')

#### СРАВНЕНИЕ МОДЕЛЕЙ PSM, IPTW, DOUBLY ROBUST

In [ ]:
# Bootstrap позволяет оценить доверительные интервалы для ATE для нерегрессионных моделей.  
# При этом довольно проблематично оценить качество построения ps, здесь еще много открытых вопросов

# ДАННЫЕ
# df = pd.read_csv('./data/smoker.csv').rename(columns={'outcome':'y'})
# df = pd.read_csv('./data/groupon.csv')[['prom_length', 'price', 'discount_pct', 
#                                         'coupon_duration', 'featured', 
#                                         'treatment', 'revenue']].rename(columns={'revenue':'y'})
df = pd.read_csv('./data/learning_mindset.csv').rename(columns={'intervention' : 'treatment',
                                                                'achievement_score' : 'y'})
X = df.drop(columns=['treatment', 'y']).columns.values
Y = 'y'
T = 'treatment'
# todo - категориальные пременные

In [ ]:
# линейная регрессия
s = 'treatment'
for j in range(0, len(df[X].columns)):
    s = s + f' + {df[X].columns[j]}'
model = smf.ols(f'y ~ {s}', data=df).fit() # OLS в R-нотации
pd.DataFrame((model.summary().tables[1]))[:4]

In [ ]:
# базовый матчинг
def based_mathing(df, X, T='treatment', Y='y'):
    cm = CausalModel(
        Y=df[Y].values, 
        D=df[T].values,  #T
        X=df[X].values
    )
    cm.est_via_matching(matches=1, bias_adj=True)
    return cm.estimates['matching']['ate']

def psm(df, X, T='treatment', Y='y'):
    df['ps'] = LogisticRegression(C=1e6).fit(df[X], df[T]).predict_proba(df[X])[:, 1] 
    cm = CausalModel(
            Y=df[Y].values, 
            D=df[T].values,  #T
            X=df.ps.values
        )
    cm.est_via_matching(matches=1, bias_adj=True)
    return cm.estimates['matching']['ate']

def iptw(df, X, T='treatment', Y='y'):
    df['ps'] = LogisticRegression(C=1e6).fit(df[X], df[T]).predict_proba(df[X])[:, 1] 
    # IPTW calc
    weight = ((df.treatment - df.ps) / (df.ps * (1 - df.ps)))
    return np.mean(weight * df.y)

# Комбинация регрессионной и IPTW модели. Дает верный прогноз если верна хотя бы одна из моделей
# часть формулы с неверной модели зануляется исходя из возникающего bias оценки (например когда OLS дает плохой предикт Y)
def doubly_robust(df, X, T='treatment', Y='y'):
    # PSM
    ps = LogisticRegression(C=1e6, max_iter=1000).fit(df[X], df[T]).predict_proba(df[X])[:, 1]
    
    # OLS estimator
    mu0 = LinearRegression().fit(df.query(f"{T}==0")[X], df.query(f"{T}==0")[Y]).predict(df[X])
    mu1 = LinearRegression().fit(df.query(f"{T}==1")[X], df.query(f"{T}==1")[Y]).predict(df[X])
    
    # ATE по комбинации двух моделей
    return (
        np.mean(df[T]*(df[Y] - mu1)/ps + mu1) -
        np.mean((1-df[T])*(df[Y] - mu0)/(1-ps) + mu0)
    )

In [ ]:
# psm - достаточно долгая процедура, лучше работает для небольших выборок
np.random.seed(88)
# ates = [psm(df.sample(frac=1, replace=True), X) for j in range(500)]
np.percentile(ates, 2.5), np.percentile(ates, 97.5)

In [ ]:
# микс IPTW + regression; работает быстро
np.random.seed(88)
ates = [doubly_robust(df.sample(frac=1, replace=True), X) for j in range(500)]
np.percentile(ates, 2.5), np.percentile(ates, 97.5)

In [ ]:
# IPTW работает относительно быстро
np.random.seed(88)
ates = [iptw(df.sample(frac=1, replace=True), X) for j in range(500)]
np.percentile(ates, 2.5), np.percentile(ates, 97.5)

In [ ]:
# ВАЛИДАЦИЯ СБАЛАНСИРОВАННОСТИ ВЫБОРКИ ПО КОВАРИАТАМ ЧЕРЕЗ СТАНДАРТИЗИРОВАННУЮ РАЗНОСТЬ
# Если разность <10%, это неплохо
def cohen_d(d1, d2): # d1,2 - проверяемая ковариата из X для 1 и 2 групп
    n1, n2 = len(d1), len(d2)
    s1, s2 = var(d1, ddof=1), var(d2, ddof=1)
    s = sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
    u1, u2 = mean(d1), mean(d2)
    return round(100 * (u1 - u2) / s, 2)

In [ ]:
# здесь проверяется PSM без коррекции на неточность совпадений через лог-регрессию
df_stat = pd.DataFrame(None, columns=['metric', 'smd%' , 'smd_abs', 'smd_after']); j=0
df['ps'] = LogisticRegression(C=1e6).fit(df[X], df[T]).predict_proba(df[X])[:, 1] # теор вероятность попасть в контроль
df_con = df[df.treatment == 0]
df_exp = df[df.treatment == 1]

smd_before_list, smd_after_list = [], []
for c in X:
    
    smd_before = cohen_d(df_con[c].values, df_exp[c].values)
    df_con['c_match'] = KNeighborsRegressor(n_neighbors=1).fit(df_exp[['ps']].values, 
                                                                        df_exp[c]).predict(df_con[['ps']].values)
    
    smd_after = cohen_d(df_con[c].values, df_con['c_match'].values)
    smd_before_list.append(smd_before)
    smd_after_list.append(smd_after)
    df_stat.loc[j, :] = c, smd_before, abs(smd), smd_after
    j+=1
print(np.mean(smd_before), np.mean(smd_after))
df_stat.sort_values(by='smd_abs', ascending=False)

#### DID (Diff-n-Diff)

https://www.kaggle.com/code/harrywang/difference-in-differences-in-python/notebook  
https://stats.stackexchange.com/questions/160359/difference-in-difference-method-how-to-test-for-assumption-of-common-trend-betw  
Метод, который используется для оценки влияния изменения на метрику в парадигме до/после.  
Мы ищем параллельный данной метрике тренд (мб из соседнего сегмента) на предыстории -  
и затем в предположении что на параллельный тренд изменение не распространяется, используем  
его в качестве сравнительного контроля (а точнее разницу между трендами до/после)

diff_in_diff = did = (y_exp_avg_after - y_exp_avg_before) - (y_control_avg_after - y_control_avg_before)  

<u> linear regression: </u>  
y = b0 + b1 * exp_group + b2 * dt + b3 * exg_group * dt  
(или y ~ exp_group + dt + exp_group * dt)  
exp_group = 1 if exp else 0  
dt = 1 if after else 0

y_control_avg_before = b0 ; y_control_avg_after = b0 + b2  
y_exp_avg_before = b0 + b1 ; y_exp_avg_after = b0 + b1 + b2 + b3  
did = (y_exp_after - y_exp_before) - (y_control_after - y_control_before) = b3  
  
В парадигме ATE: exp_group * dt = Treatment; Т.е Treatment = 1 -> exp_after; Treatment = 0 -> con_before.  
  
Приложение:  
Оценку дов. интервалов для did можно провести через бутстрап либо F-критерий коэф. b3 в рамках линейной регрессии.  
did = отклонение от параллельности тренда; если тренды остались параллельны (T не подействовал) -> p_val(b3>0) > 0.05  
  
Условия применимости:  
Работает только когда, тренды метрик групп con, exp были параллельны ДО целевого воздействия  
Это можно проверить, если брать интервалы time = x;y до time_T и убеждаясь, что b3 ~ 0


<img src="images/diff_n_diff.png" width="400" align="left">  

In [ ]:
# генерим тестовый датасет формата df = exp_group(0,1); val(metric); dt(after,before->1,0)
df1 = pd.DataFrame({'dt' : np.repeat(list(range(6)), 6*[10])})
df1['val'] = df1['dt'].apply(lambda x: x + np.random.uniform(-0.3, 0.3))
df1['exp_group'] = 0

df2 = df1.copy()
df2['val'] = df2['dt'].apply(lambda x: x + 2 + np.random.uniform(-0.3, 0.3))
df2.loc[df2.dt == 5, 'val'] = df2['val'] + 1
df2['exp_group'] = 1

df = pd.concat([df1, df2], ignore_index=True)
tm1 = df1.groupby('dt').val.mean().reset_index()
tm2 = df2.groupby('dt').val.mean().reset_index()

# как ведут себя средние выборки для двух групп con,exp с течением времени
plt.plot(tm1.dt, tm1.val); plt.plot(tm2.dt, tm2.val); plt.grid()

In [ ]:
# выбираем сравниваемые два интервала
def get_slice(df, dt1, dt2):
    tmp = df[df.dt.isin([dt1, dt2])]
    tmp.loc[tmp.dt == dt1, 'dt'] = 0; tmp.loc[tmp.dt == dt2, 'dt'] = 1
    return tmp

# расчет Dif-in-Dif аналитически
def did_calc(df):
    return (df.loc[(df.dt == 1) & (df.exp_group == 1)].val.mean() - df.loc[(df.dt == 0) & (df.exp_group == 1)].val.mean()) - \
            (df.loc[(df.dt == 1) & (df.exp_group == 0)].val.mean() - df.loc[(df.dt == 0) & (df.exp_group == 0)].val.mean())

# оценка эффекта через лин регрессию
def model_calc(df, dt_int=(0,1)):
    model = smf.ols('val ~ exp_group + dt + exp_group * dt', data=df).fit()
    return dt_int, model.params.loc['exp_group:dt'], model.pvalues.loc['exp_group:dt'], (model.conf_int().loc['exp_group:dt'][0], model.conf_int().loc['exp_group:dt'][1])

tmp = get_slice(df, 0 ,1)
print('did = ', did_calc(tmp))
print('b3_ols = ', model_calc(tmp)[1], '; p_val(b3>0) = ', model_calc(tmp)[2])

In [ ]:
df_stat = pd.DataFrame(None, columns=['dt_1_2', 'coef', 'pval', 'confint']); j=0
for dt_int in ([0,1], [1,2], [2,3], [3,4], [4,5]):
    tmp = get_slice(df, dt_int[0], dt_int[1])
    res = model_calc(tmp, dt_int)
    df_stat.loc[j,:] = res; j+=1
print('Видим что до момента 4-5 тренды были параллельными; далее произошло отклонение от тренда (см график выше)')
df_stat

In [ ]:
# покажем что бутстрап did дает тоже самое что регрессия (больше данных - ближе точность)

tmp = get_slice(df, 4, 5) # берем пример сравниваемых интервалов
print('OLS conf = ', np.round(model_calc(tmp)[3], 4))

did_list = []
for j in range(700):
    tmp_boot = tmp.sample(frac=1, replace=True) # генерим датасет с аналогичным распределением
    did_list.append(did_calc(tmp_boot))
print('BOOT conf = ', np.round(np.percentile(did_list, 2.5), 4), np.round(np.percentile(did_list, 97.5), 4))

#### SYNTHETIC CONTROL

https://matheusfacure.github.io/python-causality-handbook/15-Synthetic-Control.html - основная глава в книге  
https://medium.com/towards-data-science/causal-inference-with-synthetic-control-in-python-4a79ee636325 - доп статья

In [ ]:
# с 1980х в Калифорнии резко увеличили цену (retail price) за пачку сигарет - стремясь минимизировать курение
cigar = pd.read_csv('./data/synth_control_smoking.csv')
cigar[cigar.state == 'California'][['year', 'retprice']].plot(x='year', y='retprice')
cigar['after_treatment'] = cigar.year < 1988
cigar.drop(columns=['beer', 'lnincome', 'age15to24'], inplace=True)
# cigar[cigar.state == 'California'][['year', 'cigsale']].plot(x='year', y='cigsale')
plt.grid()

In [ ]:
plt.plot(cigar[cigar.state == 'California'].year, cigar[cigar.state == 'California'].cigsale)
tmp = cigar[cigar.state != 'California'].groupby('year').cigsale.mean().reset_index()
cigar['state'] = cigar.state.tolist()
plt.plot(tmp.year, tmp.cigsale)
plt.grid()
plt.vlines(x=1988, ymin=40, ymax=140, linestyle=":", lw=2, label="Proposition 99")
plt.title('Динамика продаж на человека до/после закона запрета в Калифорнии')
plt.legend(['california', 'not california_avg'])

По графику мы видим примерно, что в наблюдается ускорение расхождения в потреблении  
в Калифорнии и других штатах после принятия закона, однако нельзя оценить точно.  
Synthetic control - метод построения искуственного контроля (в данном случае штата),   
который ведет себя очень похоже на Калифорнию до изменений - и с большой вероятностью  
моделирует поведение после изменений как если бы их не было.

---
Ключевая идея - мы считаем, что метрику по целевому штату (Калифорния) можно разложить в базис  
вспомогательных штатов (units) на разных временных отрезках. Т.е до внесения изменения,  
можно сделать искуственный штат = комбинация вспомогательных, который будет очень близок Калифорнии.  

Матрица X - по колонкам моменты времени dt, по строкам - штаты (юниты разложения).  
Вектор Y - целевая метрика = продажи в Калифорнии во времени. 
Подбираем веса в тренировке X * w = Y.  
Добиваемся чтобы до интервенции предикт был хороший - используем экстраполяцию на период после.  

PS. Для двух переменных регрессия по векторам x1; x2 ищет коэф. чтобы y = k1 * x1 + k2 * x2  
Каждый вектор x1 = (a1 a2 a3 ... an) - можно рассматривать как набор координат во времени.  
Тогда для каждого "момента времени" y_t = k1 * x1_t + k2 * x2_t и.т.д  
Найдем k1; k2 - можем экстраполировать y_t_next по имеющимся x1_t_next; x2_t_next в будущем.  
Мы через регрессию ищем k1; k2; ... kn для n юнитов разложения (т. е n штатов)  

Важно также помнить о том что при большом кол-ве юнитов (aka предикторов)  
линейная регрессия легко переобучается -> нужна регуляризация.

In [ ]:
dfs = pd.pivot_table(cigar, 
                    index='year', 
                    columns='state', 
                    values=['cigsale'], 
                    aggfunc=np.mean)['cigsale'].reset_index()

df = dfs[dfs.year <= 1988].copy() # train
y = df['California'].values
X = df.drop(columns='California').values

# обучаем регрессию - получаем коэффициенты k1;k2... для каждого штата
# model_ = LinearRegression(fit_intercept=False).fit(X, y) # без регуляризации - переобучение
model_ =  Lasso(fit_intercept=False, alpha=1.0).fit(X, y) # регуляризация по дефолту

# экстраполируем данные на весь период времени
synth = model_.predict(dfs.drop(columns=['California']).values)
real = dfs.California.values
plt.plot(dfs.year.values, real)
plt.plot(dfs.year.values, synth)
plt.grid()
plt.vlines(x=1988, ymin=40, ymax=140, linestyle=":", lw=2, label="Proposition 99")
plt.title('Построение синтетического контроля \n через линейную регрессию с регуляризацией')
plt.legend(['Калифорния real', 'Калифорния synth'])

Проблема регрессии - она экстраполирует, не учитывая "схожесть" регионов друг с другом.  
Например, один из весов модели может быть -100 и уносить показатели одного из штатов в несущ. область.  
Чтобы это избежать - можно находить оптимальные веса оптимизацией (метод взвешенных средних),  
накладывая смысловые ограничения вроде - **сумма всех весов равна 1, веса положительные**.  
Это будет лучше учитывать общую специфику и <u>изначальную схожесть</u> данных в используемом базисе.  
Теперь физический смысл весов - нечто схожее с вероятностью.

In [ ]:
# Реализация метода взвешенных средних через scipy оптимизацию
from typing import List
from operator import add
from toolz import reduce, partial
from scipy.optimize import fmin_slsqp
def loss_w(W, X, y) -> float:
    return np.sqrt(np.mean((y - X.dot(W))**2))
def get_w(X, y):
    w_start = [1/X.shape[1]]*X.shape[1]
    weights = fmin_slsqp(partial(loss_w, X=X, y=y),
                         np.array(w_start),
                         f_eqcons=lambda x: np.sum(x) - 1,
                         bounds=[(0.0, 1.0)]*len(w_start),
                         disp=False)
    return weights
calif_weights = get_w(X, y)
# строим
synth = dfs.drop(columns=['California']).values.dot(calif_weights)
real = dfs.California.values
plt.plot(dfs.year.values, real)
plt.plot(dfs.year.values, synth)
plt.grid()
plt.vlines(x=1988, ymin=40, ymax=140, linestyle=":", lw=2, label="Proposition 99")
plt.title('Построение синтетического контроля \n через метод взвешенных средних')
plt.legend(['Калифорния real', 'Калифорния synth'])

#### RANDOM SPLIT TESTING (ABn)

https://habr.com/ru/users/nnazarov/ - полезный цикл статей

---
Как описывали выше, при рандомизированном разбиении выборки ничего не влияет на T.  
В результате ATE = ATT; Влияние конфаундеров исчезает.  
При этом остаются ряд других проблем связанных с оценкой эффекта через статистические тесты:  

1. Проблема точности оценки ATE и связанная с ней длительность теста. См подробнее раздел по MDE 
2. Проблема применимости стат-теста к конкретной метрике (см подробнее раздел про Валидацию, сбалансированность)
3. Проблемы множественных сравнений и работы с Ratio-метриками (линеаризация)

##### Variance reduction. CUPAC, CUPED, Stratification

https://medium.com/glovo-engineering/variance-reduction-in-experiments-using-covariate-adjustment-techniques-717b1e450185  
https://github.com/Glovo/covariate-adjustment-blogpost/tree/main  
https://habr.com/ru/companies/X5Tech/articles/780270/

---
Как обсуждали в разделе "Линейная регрессия" - в системе Y ~ T + X учет ковариат X, которые НЕ влияют на T  
позволяет снижать дисперсию метрики Y -> увеличивать точность оценки ATE (по сути коэф. перед T = treatment).   
Но это можно рассмотреть и так: Y - X = Y_adj ~ T; Y_adj = скорректированная по дисперсии метрика к Y.  
Основные условия:
1. avg(Y) = avg(Y_adj)
2. std(Y_adj) < std(Y)
3. Y ~ Y_adj (сонаправленность метрик)
  
Пусть у нас есть ковариата X, для которой cov(X, Y) > 0 при этом cov(X, T) = 0  
Тогда согласно п.3: Y_predict = Yp = a0 + tetta * X , по методу наименьших квадратов tetta = cov(X,Yp) / var(X)  
Рассмотрим Y_adj = Y - Yp + avg(Yp):
видно что avg(Y_adj) = avg(Y)  
т.к X объясняет часть вариации Y -> std(Y_adj) < std(Y)  
Y_adj ~ Y по методу построения Yp.  (К слову, рандомная X не подходит, так как будет плохо предсказывать Y в регрессии)
  
С точки зрения длительности AB: N ~ var(X)/mde^2(X) -> уменьшая дисперсию метрики, можно пропорционально снижать N.
  
<u>CUPED = Controlled-experiment Using Pre-Experiment Data</u>  
Если в качестве X использовать Y_prev = значения метрики на пред-экспериментальном периоде (до влияния T), то  
Y_adj = Y_cuped = Y - Yp + avg(Yp) = Y - tetta * (Y_prev - avg(Y_prev))  
Пре-экспериментальный период никак не может повлиять на T -> условие выполняется.  
Если брать не очень большой период (например, последние 2 недели) -> достигается адекватная cov(Y, Y_prev).  
  
<u>Мультиковариатная регрессия</u>  
CUPED - частный случай, когда мы используем Y_prev как ковариату-предиктор для Y в уравнении Y ~ T + Y_prev  
Если мы знаем другие предикторы, уточняющие поведение Y, которые НЕ влияют на T, то Y ~ T + X1 + X2 + ...  
Учет таких предикторов повышает точность оценки ATE (см ниже пример в коде)
  
<u>CUPAC = Control Using Predictors as Covariates</u>  
CUPAC - расширение над уравнением регрессии с использованием ML, когда для снижения вариации целевой метрики Y  
используются модели вроде случ. лесов итд.  
Идея: имеем ковариаты X1, X2, ... Y_prev -> обучаем модель ML(X1, X2 .., Y_prev) -> Y_forecast.  
Используем Y_forecast как ключевую ковариату в уравнении Y ~ T + Y_forecast.  
PS. CUPED - частный случай CUPAC с одной ковариатой Y_prev   
PSS. Мультирегрессия - частный случай CUPAC где ML-модель - линейная регрессия.  
<u>Во всех</u> случаях Y_adj = Y - tetta * (Y_forecast - avg(Y_forecast))  
std(Y_adj) < std(Y); avg(Y_adj) = avg(Y)  
    
<u>Подбор ковариат и пред-тестового периода</u>      
Главное условие: ковариаты Y_prev, X1, X2, ... - никак не влияют на распределение T и обратно: cov(Y_forecast, T) = 0  
Поэтому их берут на пред-тестовом периоде, чтобы Y_forecast = ML.fit(X1, ...) точно не зависело от последующего T.  
   
При выборе пред-тестового периода можно использовать знания о сезонности (искать аналогичный сезон в прошлом).  
Размер пред-тестового периода может варьироваться (от exp_duration и больше)
Подбор пред-тестового периода можно совершать, каждый раз оценивая насколько Y_adj(Y_forecast) снижает дисперсию относительно Y. 
  
CUPAC/CUPED также можно использовать для бинарных метрик (вроде конверсии) - он будет улучшать чувствительность оценок.  

In [ ]:
# генерация данных
n = 5000
data = pd.DataFrame({
    'T': np.random.binomial(1, 0.5, size=n),
    'Y': np.random.normal(size=n),
    'Y_prev': np.random.normal(size=n),
    'Cov2': np.random.normal(size=n),
})
effect = 2

# пре-тестовый период (нет зависимости T так как нет еще теста)
data_prev = data[:2500] 
# data_prev['Y'] += 3 * data_prev['Y_prev'] + 4 * data_prev['Cov2']
data_prev['Y'] += 3 * data_prev['Y_prev'] + 4 * data_prev['Cov2']**2


# тестовый период
data_now = data[2500:]
# data_now['Y'] += effect * data_now['T'] + 3 * data_now['Y_prev'] + 4 * data_now['Cov2'] 
data_now['Y'] += effect * data_now['T'] + 3 * data_now['Y_prev'] + 4 * data_now['Cov2']**2 


# реализация CUPED
# в AB тесте оценку tetta можно проводить как по now, так и по prev данным.
# если размеры AB групп достаточно велики, лучше оценивать tetta на объединённых данных контрольной и экспериментальной групп. 
# Если размеры групп малы, то оценка tetta на исторических данных может дать лучшее качество.
def get_cuped_adj(Y, Y_prev):
    # Y = целевая метрика; Y_prev эта же метрика на клиентах на пред-тестовом периоде
    tetta = np.cov(Y, Y_prev)[0][1] / np.var(Y_prev)
    return Y - (Y_prev - np.mean(Y_prev)) * tetta

# оценка ATE через регрессию
def get_est(formula, df, text):
    model = smf.ols(formula, data=df).fit()
    conf = np.round(model.conf_int().loc['T'], 3); std = round(model.bse.loc['T'], 3)
    print(f'{text}: ate = {conf[0]} - {conf[1]}; std_ate = {std}')
    
# Пример использования CUPAC: получаем y_forecast = ML(X1, X2 ...) через древесный регрессом; корректируем Y
def get_cupac_adj(X_past, Y_past, X, Y):
    # X_past, Y_past - фичи и целевая метрика на предэкспериментальном периоде
    # X, Y - фичи и метрика в эксперименте - для предикта Y_forecast и поправки
    gbm = HistGradientBoostingRegressor().fit(X_past, Y_past)
    Y_forecast = gbm.predict(X)
    return get_cuped_adj(Y, Y_forecast)


Y_cuped = get_cuped_adj(data_now.Y, data_now.Y_prev)
Y_cupac = get_cupac_adj(
                      data_prev[['Y_prev', 'Cov2']].values, # целевая метрика пред-тестового периода
                      data_prev.Y.values, # целевая метрика пред-тестового периода
                      data_now[['Y_prev', 'Cov2']].values,
                      data_now.Y.values
                     )

data_now['Y_cuped'] = Y_cuped
data_now['Y_cupac'] = Y_cupac

print(f'реальный ATE = {effect}')
get_est('Y ~ T', data_now, 'без редукции VAR')
get_est('Y_cuped ~ T', data_now, 'CUPED - формула')
get_est('Y ~ T + Y_prev', data_now, 'CUPED, как частный случ. мультиковар. регрессии')
get_est('Y ~ T + Y_prev + Cov2', data_now, 'Мультиковар. регрессия')
get_est('Y_cupac ~ T', data_now, 'CUPAC обученный на prev-периоде')

<u>STRATIFICATION</u>  
https://habr.com/ru/companies/yandex/articles/497804/  
https://habr.com/ru/companies/X5Tech/articles/596279/  
https://habr.com/ru/companies/X5Tech/articles/826488/  

---
<u>Ключевая идея</u>    
заменяем среднее для метрики X на средневзвешенное по стратам.
Страта - группа с одинаковым значением S = комбинированной ковариаты страты.  
  
Например, Y - сумма покупок компаний; X - сегмент бизнеса, возраст компании; S = {SMB_new, SMB_old, Large_new, ....}.  
При этом SMB_new, SMB_old ... это конкретные страты, по которым будем взвешивать среднее.  
Для составления вектора S можно использовать как простую конкантенацию так и kNN методы.  
  
Веса каждом страты W_s = len(X in S) / len(X) = вероятности, оцениваются на пред-экспериментальном периоде.  
Среднее тогда avg_strat(X) = sum(W_s * avg(X in S)); std_strat_X < std_X.  
Причина снижения дисперсии - фильтрация меж-стратовой дисперсии (см "Статистика") - остается только внутри-стратовая.  
Снижение тем выше - <u>чем сильнее разница средних между стратами</u> (то есть выше межгрупповая дисперсия)
  
Важное условие: cov(S, T) = 0, метрика S не влияет на экспериментальную разбивку, существует до и после начала теста.  
Сам тест также не влияет на метрику S.
  
Стратификация классическая и пост-стратификация:
1. Классическая - когда мы семплируем A/B группы следя за равномерным наполнением страт и затем применяем avg_strat_ttest  
2. Пост - когда мы семплируем без контроля равномерности страт (как обычно), но далее также применяем avg_strat_ttest  
Различия в оценках std методами 1 и 2 стремятся к нулю при большом кол-ве данных (>1e3) -> обычно можно идти по пути 2.
  
Выбор признаков для S можно осуществлять на исторических данных исходя из условия:  
максимальная меж-групповая дисперсия среднего целевой метрики посчитанной по разным стратам.  

---
СUPAC vs Stratification.  
В большинстве случаев, если использовать S в CUPAC для предикта Y_forecast, то снижение  
дисперсии тотал метрики за счет обнуления межгрупповой дисперсии уже будет учтено в алгоритме.  
Поэтому использование стратификации поверх CUPAC не даст значительного улучшения.  
  
Исключение - если после старта эксперимента проихошло <u>не зависящее от эксперимента</u>  влияние на метрику Y  
конкретно в каких-то стратах (например, сезонный рост покупок в сегменте LARGE которого не было на пред-периоде).  
В этом случае, Y_forecast не учтет выросшую в тесте межгрупповую дисперсию, а методы стратификации поверх CUPAC  
смогут ее контролировать. Поэтому в общем случае полезно использовать оба метода.  

In [ ]:
# сгенерируем тестовые данные; Y будет зависеть от значения страт
# чем больше разброс id страт - тем больше разброс Y_strat - лучше работает стратификация
size = 5000
effect = 0.01
std = 3
strat_seq = [1, 2, 5]

# распределение страт на пред-экспериментальном периоде
df_prev = pd.DataFrame({'S' : np.random.choice(strat_seq, size=size)})
weights = df_prev.S.value_counts(normalize=True) # series

df_con = pd.DataFrame({'S' : np.random.choice(strat_seq, size=size)})
df_con['Y'] = df_con['S'].apply(lambda x: np.random.normal(x**2, std))

df_test = pd.DataFrame({'S' : np.random.choice(strat_seq, size=size)})
df_test['Y'] = df_test['S'].apply(lambda x: np.random.normal(x**2 + effect, std))

from help_tools import ttest_calc, ttest_stratification_calc
conf1 = ttest_stratification_calc(df_con, df_test, weights)[0] # пост стратификация
conf2 = ttest_calc(df_con.Y, df_test.Y)[0] # классический тест
print(f"""conf_width: classic test = {round(conf2[1] - conf2[0], 4)}; stratification = {round(conf1[1] - conf1[0], 4)}""")

##### Длительность теста и MDE 

MDE = Minimum Detectable Effect = минимальное изменение которое будет детектируемо через стат-тест с  
выбранными уровнями ошибки первого и второго рода.  
  
В случае с T-тестом (см страничку "Статистика") оцениваем delta = avg(control) - avg(experiment).  
Дисперсия средневыборочного std_X_mean = std(X)/sqrt(N), N - длина выборки.  
Поэтому мин. детектируемая delta средних: mde = T(alpha, power) * std(X) / sqrt(N).  
  
Отсюда следует, что N (кол-во клиентов aka длительность теста) ~ var(X) / mde^2.  
Чем меньше var(X) = шум метрики - тем меньше данных надо для фиксированной точности.  
Кроме того с ростом ожидаемого mde квадратично падает N
  
1) Учитывая в дизайне АБ теста рост mde - можно квадратично ускорять его.  
Пример: замеряем как изменится рост выручки X от роста цены на 10%.  
Если запустить тест в формате (уменьшили цену на 10%) VS (увеличили цену на 10%), то  
ожидаемая дельта эффекта удвоится -> данных надо будет гораздо меньше.  
  
2) Другой вариант - сокращать var(X) - см подробнее в разделе "Variance reduction"  

Приложение:  
При оценке mde эксперимента на контрольной выборке, полагаем что std_experiment = std_control.  
В будущем, на накопленных данных control/experiment можно посчитать mde_real(control, experiment).  
Тогда в формуле std будет учитываться как sqrt(std_control^2 + std_experiment^2)

In [ ]:
from help_tools import ttest_calc, get_mde_detail
control, test = np.random.normal(1, 1, 1000), np.random.normal(1.2, 1, 1000)
ttest_calc(control, test) # confint; pvalue

In [ ]:
# пример оценки MDE
control = np.random.normal(1, 1, 1000)
get_mde_detail(control)

##### Сбалансированность и репрезентативность

1. Репрезентативность - обобщаемость используемой выборки на всю генеральную совокупность.   
При нестационарном поведении генеральной совокупности во времени необходимо случайно  
семплировать выборку для теста также по времени.   
2. Сбалансированность - условия по возможности попадания семплов в выборку в тесте/контроле должны быть одинаковые  
Перекос в размере выборки относительно ожидаемой показывает на ошибкув рандомизации.  
  
---
Тест на сбалансированность через хи-квадрат:  
  
Критерий соответствия эмпирических частот ожидаемым.  
При сравнении номинативных признаков образуется таблица сопряженности с совместными частотами.  
Она же образуется, если сравниваем теоретическое и реальное распределения вероятностей.  
Частный случай таблицы - вектор эмпирических частот признака (например частоты орла/решки).  

H0-гипотеза: наблюдаемые частоты совпадают с ожидаемыми (теоретическими).  
В этом случае $\chi^2 = \sum_{ij}{(Obs - Exp)^2 / Exp}$ -- расстояние Пирсона.  
Для таблицы размерностью ij критерий будет удовлетворять df = (i-1)(j-1) степень свободы.  
Так как мы знаем тотал значения таблицы и значит одна степень свободы всегда фиксирована.  

PS. Величина Obs распределена биномиально (вероятность попасть в ячейку j Obs раз).  
При малых вероятностях получаем Пуассона, где std^2=mean=Exp при нулевой гипотезе. Тогда t-score  
(Obs - Mean) / Std ~ (Obs - Exp)/sqrt(Exp) ~ N(0, 1).

In [ ]:
# Оценка сбалансированности выборок в АБ тесте
from help_tools import check_branch_balance
f_real = [100, 110, 105]
print('Три ветки, ожидаемое распределение равномерно: ', check_branch_balance(f_real))
f_real = [85, 20]; f_exp = [80, 20]      
print('Две ветки, катим тест 80/20: ', check_branch_balance(f_real, f_exp))
f_real = [100, 20]; f_exp = [80, 20]      
print('Две ветки, катим тест 80/20: ', check_branch_balance(f_real, f_exp))

##### Множественные сравнения

https://habr.com/ru/articles/772940/
  
При проведении A/B/C... = N тестов могут возникнуть такие задачи:
1. Определить ветку-победителя. Потребуется сделать K = N * (N-1) / 2 стат-тестов, проверяя каждую пару веток.
2. Найти хотя бы одну ветку, лучшую чем контроль. Здесь требуется совершить N сравнений с контролем.  
3. Остановить тест если хотя бы одна из K значимых метрик прокрасится отрицательно (даже в простом A/B тесте)

Проблема в контроле ошибки первого рода: если не использовать поправки на множественное сравнение,  
то например вероятность найти хотя бы один прокрас FWER = 1 - (1 - alpha)^N ~ alpha * N >> alpha 

В зависимости от консервативности контроля выделяют два типа множественных корректировок:

1. Контроль FWER = family-wise error rate = вероятность совершить хотя бы одну ошибку при проверке X тестов.  
Хотя бы одну ошибку сделали = сделали неправильное решение на базе группы.  
Считаем любую ошибку критичной и ожидаем ее появление в небольшое кол-ве случаев.  
Используют поправку Бонферрони (alpha_adj = alpha / N); поправку Холма (более мощная), Тьюкки итд.  
По мощности оптимально использовать метод Холма

2. Контроль FDR = false discovery rate = доля всех ложных прокрасов среди тестов, которые <u>прокрасились</u>  
Здесь допускаем чтобы из K прокрашенных тестов у нас было alpha% прокрашенных по ошибке.  
Подходит, когда фильтруем интересные инсайты и допускаем наличие среди них доли ложно-прокрашенных.  
Мощность в среднем, очевидно, выше чем при контроле FWER
Здесь по мощности оптимально использовать метод Бенджамини-Хохберга  

Чем эффективнее идет контроль FWER (шанс хотя бы одного ложного прокраса), тем хуже чувствительность, например  
метрика FWER-II (шанс хотя бы одного случая, когда не заметили реальный прокрас).  

In [ ]:
fwer_list, tn_list = [], []
fwer_list2, tn_list2 = [], []
fwer_list3, tn_list3 = [], []
fwer_list4, tn_list4 = [], []

def get_effect(real_effect, p_val_list):
    fp = [] # ошибка первого рода
    tn = [] # ошибка второго рода
    for idx, p_val in enumerate(p_val_list):
        if real_effect[idx] == 0 and p_val < 0.05: # FP
            fp.append(1); tn.append(0)
        elif real_effect[idx] == 0 and p_val >= 0.05: # FN
            fp.append(0); tn.append(0)
        elif real_effect[idx] == 1 and p_val < 0.05: # TP
            fp.append(0); tn.append(0)
        elif real_effect[idx] == 1 and p_val >= 0.05: # TN
            fp.append(0); tn.append(1)
            
    fwer = max(fp) # хотя бы один ложный прокрас в группе
    fwer_II = max(tn) # не нашли хотя бы один реальный прокрас в группе
    return fwer, fwer_II

for _ in range(1000):
    
    # генерируем одинаковые ветки
    test_list = []
    for _ in range(3): # кол-во веток
        test_list.append(np.random.normal(1, 1, 200)) # mu, std, size
        
    # добавляем ветку с реальным эффектом
    eff_delta = 0.3
    test_list.append(np.random.normal(1 + eff_delta, 1, 200))
    idx_last = len(test_list) - 1
    

    # попарное вычисление тестов
    p_val_list = []
    real_effect = []
    for idx, test in enumerate(test_list):
        for idx2, test2 in enumerate(test_list):
            if idx > idx2:
                p_val_list.append(stats.ttest_ind(test, test2)[1])
                
                # помечаем если имеется реальное различие в тестах
                if idx == idx_last or idx2 == idx_last:
                    real_effect.append(1)
                else:
                    real_effect.append(0)
                
    # корректировки p_val для множественного сравнения
    p_val_bonf = multipletests(p_val_list, alpha = 0.05, method='bonferroni')[1] # holm
    p_val_holm = multipletests(p_val_list, alpha = 0.05, method='holm')[1] # holm
    p_val_bh = multipletests(p_val_list, alpha = 0.05, method='fdr_bh')[1] # Benjamini/Hochberg 

    # принятие решения по группе - расчет коллективного ложного прокраса и ошибки 2-го рода
    fwer, fwer_II = get_effect(real_effect, p_val_list)
    fwer_list.append(fwer); tn_list.append(fwer_II)
    
    fwer, fwer_II = get_effect(real_effect, p_val_bonf)
    fwer_list2.append(fwer); tn_list2.append(fwer_II)
    
    fwer, fwer_II = get_effect(real_effect, p_val_holm)
    fwer_list3.append(fwer); tn_list3.append(fwer_II)
    
    fwer, fwer_II = get_effect(real_effect, p_val_bh)
    fwer_list4.append(fwer); tn_list4.append(fwer_II)

# статистики для разных подходов
print(f'No correction. FWER = {np.mean(fwer_list)}; FWER-II = {round(np.mean(tn_list), 5)}')
print(f'Bonferroni-FWER correction. FWER = {np.mean(fwer_list2)}; FWER-II = {round(np.mean(tn_list2), 5)}')
print(f'Holm-FWER correction. FWER = {np.mean(fwer_list3)}; FWER-II = {round(np.mean(tn_list3), 5)}')
print(f'BH-FDR correction. FWER = {np.mean(fwer_list4)}; FWER-II = {round(np.mean(tn_list4), 5)}')

##### Валидация

https://habr.com/ru/companies/hh/articles/321386/  
https://habr.com/ru/companies/X5Tech/articles/706388/

---
Когда мы работаем с метрикой X и планируем оценивать ее изменение в AB тесте, используя статистический тест,  
нужно перед этим убедиться, что пройдена валидация метрики для этого теста.  
  
Валидация:
убеждаемся, что при выбранной для теста ошибке первого рода - на группе АА-тестов доля ложных  
прокрасов (FPR) соответствует этой ошибке.  
PS. В общем случае через АА тест можно проверить любой тип теста (Манна-Уитни, bootstrap итд)    
Подробнее рассмотрим T-test на средних, как наиболее часто используемый.  
  
Ситуация, когда метрика не проходит валидацию:
1. Выбросы в метрике -> не выполняется ЦПТ -> средневыборочное не распределено нормально (см "Статистика")
2. Семплы зависимы -> не выполняются классические операции над вероятностями -> нарушение нормальности
3. Слишком мало данных
  
При валидации T-теста, мы разбиваем выборку на случайные пары АА-фрагментов, вычисляем p_val и оцениваем  
долю ложных прокрасов с доверительным интервалом.  
Дополнительно можно посмотреть на нормальность средневыборочного (необходимое условие валидации, но не достаточное)

In [ ]:
from help_tools import validation_ttest
# пример адекватной для T-теста метрики
X = np.random.normal(1, 1, 1000) 
ans, p_val_list, avg_list = validation_ttest(X)
print(ans)

In [ ]:
# пример метрики с дополнительными "выбросами"
X = np.append(np.random.normal(1, 1, 1000), np.random.normal(1000, 1, 9)) 
ans, p_val_list, avg_list = validation_ttest(X)
print(ans)

##### Последовательный анализ, бакетный и децильный анализы TODO

In [ ]:
# todo
# https://www.youtube.com/watch?v=p_5YzShN4sg
# децильное распределение val_list1, val_list2 на основании бутстрапа
# N-й дециль -> доверительный интервал , контрольное значение

##### Ratio-метрики

https://habr.com/ru/companies/X5Tech/articles/740476/
https://habr.com/ru/company/avito/blog/454164/

---

Метрика типа R = X / Y (например средняя длительность сессии на сайте по всем клиентам).  
R = (x1 + x2 + ...) / (y1 + y2 + ...)  
В общем случае x1, x2, ... зависимые, так как один клиент может создавать несколько сессий и пр.  
Поэтому, нельзя работать с параметрическими тестами вроде t-test.  
(зависимость -> не работают теоремы сложения/умножения вероятностей -> не работают все распределения).  
  
1. Чтобы начали работать классические тесты - необходимо превратить Ratio-метрику в поюзерную  
R = X / Y = alpha * X + betta * Y + gamma  
Поюзерная метрика -> пренебрегая зависимостью между юзерами -> подходит под i.i.e
Один из вариантов такого разложения - ряд Тейлора в точке avg(X)/avg(Y). 
Можно разложить до первого члена (линеаризация) или взять больше точности и след члены ряда.
  

2. Можно использовать bootstrap для семплирования Ratio  
Для него нет проблем с тем что распределение не нормальное, но это вычислительно дорого  
  
  
3. Есть дельта-метод оценки дисперсии метрики X/Y (см статью)  
Он считается быстро и удобен в оценке критериев, но не дает поюзерную метрику с которой потом удобно работать,  
например увеличивая чувствительность и пр.

In [ ]:
# семплируем отдельных "юзеров" по индексам
def ratio_bootstrap_calc(num_list1, denum_list1, num_list2, denum_list2, iter_=10**4, alpha=0.05):
    boot_list = []
    for _ in range(iter_):   
        idx = np.random.choice(np.range(len(num_list1)), len(num_list1), replace=True)
        idx2 = np.random.choice(np.range(len(num_list2)), len(num_list2), replace=True)
        num_list1_, denum_list1_ = num_list1[idx], denum_list1[idx]
        num_list2_, denum_list2_ = num_list2[idx2], denum_list2[idx2]
        delta = sum(num_list1_) / sum(denum_list1_) - sum(num_list2_) / sum(denum_list2_)
        boot_list.append(delta)
    # confint 
    return np.quantile(boot_list, [alpha/2, 1 - alpha/2])

# превращаем метрику в поюзерную раскладывая в ряд тейлора до первого члена
def ratio_taylor(metric_num, metric_denum):
    """
    X/Y ~ mx/my + 1/my * (X - mx) - mx/my**2 * (Y - my) = mx/my + 1/my(X - Y*mx/my)
    https://habr.com/ru/company/avito/blog/454164/
    """
    mx, my = np.mean(metric_num), np.mean(metric_denum)
    return mx / my + (1 / my) * (metric_num - metric_denum * (mx / my))